# Day 18 · NULL — Ghost Buses

**Question.** The timetable promises a bus. How many scheduled CTA buses never show up, and where?

**Approach.** For every weekday in October 2026 that the archive holds, the trips CTA's schedule (GTFS)
put on each bus route are compared with the trips CTA Bus Tracker actually reported on that route. Bus
Tracker positions come from the Ghost Buses archive (Chi Hack Night, now kept by the Mansueto
Institute), which saves every bus's position every five minutes. A scheduled trip with no bus behind it
is a NULL: the timetable row exists, the real-time row does not. On the plate each missing bus is one
hollow ring on its route.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Bus Tracker archive, one file per weekday (cached); trips seen per route and day |
| 3. Validate | Archive columns, days, routes; the GTFS calendar covers the window |
| 4. Transform | Scheduled vs seen per route and day; blank days; one shape per route |
| 5. Analyze | Missing buses by route, express vs local, south vs north of Roosevelt |
| 6. QA | Daily shares; routes |
| 7. Export | `plate.json` |

Run top to bottom from this folder. The tally of trips seen (`data/raw/bus_tracker_trips_*.csv`) is
committed; the daily archive files (about 44 MB each) are cached in `data/processed/rt/` and only
needed with `REFRESH = True`.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from atlas import Ledger, write_plate
from atlas.boundary import simplify
from atlas.gtfs import service_ids
from atlas.grid import to_address

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day18')
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = rebuild the tally from the daily archive files (downloads any missing)

DAY = 18
ARCHIVE = 'https://d2v7z51jmtm0iq.cloudfront.net/cta-stop-watch/full_day_data/{}.csv'
ARCHIVE_INFO = 'https://github.com/chihacknight/chn-ghost-buses/blob/main/data_analysis/README.md'
GTFS_URL = 'https://www.transitchicago.com/developers/gtfs/'
BUSTRACKER_URL = 'https://www.transitchicago.com/developers/bustracker/'
WINDOW = (date(2026, 10, 1), date(2026, 10, 27))  # October weekdays; the archive's last file at pull time
PULLED = date(2026, 11, 22)

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
RT_DIR = DAY_DIR / 'data' / 'processed' / 'rt'
TALLY = RAW_DIR / 'bus_tracker_trips_2026-10.csv'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
GTFS_ZIP = REPO / 'reference' / 'raw' / 'gtfs' / 'cta_gtfs_2026-11-01.zip'  # calendars 2026-09-20 to 2026-11-30
assert (DAY_DIR / 'day18.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

# Analysis constants ---------------------------------------------------------------------------
BLANK = 0.5          # a route-day with under half its scheduled trips seen: the record went blank
ROOSEVELT = -1200    # address units: 1200 S, the south / north split
REF_DAY = date(2026, 10, 14)  # a Wednesday in the window, for route shapes and stop departures
SIMPLIFY = 10        # Douglas-Peucker tolerance for shapes, address units (about 66 ft)

## 2. Pull

Each archive file holds every position Bus Tracker reported that day, scraped every five minutes
(`vid`, `rt`, `origtatripno`, …). A trip is one `origtatripno` on one route. Its ids do not match the
GTFS trip ids, so trips are counted, not matched: trips seen per route per day.

In [ ]:
weekdays = [WINDOW[0] + timedelta(n) for n in range((WINDOW[1] - WINDOW[0]).days + 1)]
weekdays = [d for d in weekdays if d.weekday() < 5]


def fetch_day(d: date) -> Path | None:
    """Cached archive file for a day, downloading it if needed; None if the archive has no file."""
    path = RT_DIR / f'{d.isoformat()}.csv'
    if not path.exists():
        r = requests.get(ARCHIVE.format(d.isoformat()), timeout=600)
        if r.status_code in (403, 404):
            log.warning('No archive file for %s (%s)', d, r.status_code)
            return None
        r.raise_for_status()
        RT_DIR.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    return path


if REFRESH or not TALLY.exists():
    rows = []
    for d in weekdays:
        path = fetch_day(d)
        if path is None:
            continue
        rt = pd.read_csv(path, dtype=str, usecols=['rt', 'origtatripno', 'vid'])
        g = rt.groupby('rt').agg(trips_seen=('origtatripno', 'nunique'), positions=('vid', 'size')).reset_index()
        rows.append(g.assign(day=d.isoformat()))
    tally = pd.concat(rows)[['day', 'rt', 'trips_seen', 'positions']].rename(columns={'rt': 'route'})
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    tally.to_csv(TALLY, index=False)
tally = pd.read_csv(TALLY, dtype={'route': str})
days = sorted(tally['day'].unique())
missing_days = sorted(set(d.isoformat() for d in weekdays) - set(days))
log.info('%d weekdays in the window, %d in the archive; missing: %s', len(weekdays), len(days), missing_days)
tally.head()

## 3. Validate

In [ ]:
z = zipfile.ZipFile(GTFS_ZIP)
cal = pd.read_csv(z.open('calendar.txt'), dtype=str)
covers = (cal['start_date'] <= WINDOW[0].strftime('%Y%m%d')) & (cal['end_date'] >= WINDOW[1].strftime('%Y%m%d'))
assert covers.sum() >= 40, 'GTFS calendar does not cover the window'
routes_gtfs = pd.read_csv(z.open('routes.txt'), dtype=str)
bus = routes_gtfs[routes_gtfs['route_type'] == '3'].set_index('route_id')
bus['route_long_name'] = bus['route_long_name'].str.strip()
trips = pd.read_csv(z.open('trips.txt'), dtype=str)
trips = trips[trips['route_id'].isin(bus.index)]

assert len(days) >= 15, 'Too few archive days'
per_day = tally.groupby('day')['route'].nunique()
assert per_day.min() >= 115, f'A day with too few routes seen: {per_day.idxmin()}'
assert tally['trips_seen'].ge(0).all()
unknown = sorted(set(tally['route']) - set(bus.index))
log.info('Routes seen in Bus Tracker but not in the GTFS bus routes: %s', unknown)

## 4. Transform

- **Scheduled.** GTFS bus trips whose service runs that day (calendar and exceptions), per route.
- **Seen.** Distinct trips Bus Tracker reported on the route that day, capped at the scheduled count
  (extra trips, such as added runs, do not offset missing ones).
- **Never seen.** Routes scheduled on every day of the window and never seen once are left out: the
  feed does not report them under their own number, so their buses cannot be counted.
- **Blank days.** A route-day with under half its trips seen is a gap in the record (Bus Tracker or
  the archive went blank for that route, often for days in a row), not missing buses, and is left out.
- **Route figures.** The average weekday over the route's remaining days.

In [ ]:
sched_rows = []
for d in days:
    ids = service_ids(GTFS_ZIP, date.fromisoformat(d))
    n = trips[trips['service_id'].isin(ids)].groupby('route_id').size()
    sched_rows.append(pd.DataFrame({'day': d, 'route': n.index, 'sched': n.values}))
sched = pd.concat(sched_rows)
rd = sched.merge(tally[['day', 'route', 'trips_seen']], on=['day', 'route'], how='left').fillna({'trips_seen': 0})
rd['seen'] = np.minimum(rd['trips_seen'], rd['sched'])
rd['ratio'] = rd['seen'] / rd['sched']

never = rd.groupby('route').filter(lambda g: (g['seen'] == 0).all() and len(g) == len(days))['route'].unique()
never = sorted(never)
rd = rd[~rd['route'].isin(never)]
rd['blank'] = rd['ratio'] < BLANK
log.info('Never seen: %s; route-days %d, blank %d on %d routes', never, len(rd), rd['blank'].sum(),
         rd.loc[rd['blank'], 'route'].nunique())
kept = rd[~rd['blank']]
r = kept.groupby('route').agg(sched=('sched', 'mean'), seen=('seen', 'mean'), days=('day', 'size'))
r['blank'] = rd[rd['blank']].groupby('route').size().reindex(r.index).fillna(0).astype(int)
r['miss'] = r['sched'] - r['seen']
r['share'] = r['miss'] / r['sched']
r['name'] = bus['route_long_name'].reindex(r.index)
r['express'] = r['name'].str.contains('Express|Jump')
r.sort_values('miss', ascending=False).head(10)

In [ ]:
# One shape per route (as on Plate II): per direction, the longest shape carrying at least a quarter of
# that direction's trips on the reference day; then the longer of the two directions.
ref = trips[trips['service_id'].isin(service_ids(GTFS_ZIP, REF_DAY)) & trips['route_id'].isin(r.index)]
shapes = pd.read_csv(z.open('shapes.txt'), dtype={'shape_id': str})
shapes = shapes[shapes['shape_id'].isin(set(ref['shape_id']))].sort_values(['shape_id', 'shape_pt_sequence'])
shape_len = shapes.groupby('shape_id')['shape_dist_traveled'].max()


def representative(g: pd.DataFrame) -> str:
    best = []
    for _, d in g.groupby('direction_id'):
        counts = d['shape_id'].value_counts()
        common = counts[counts >= 0.25 * counts.sum()].index
        best.append(max(common, key=lambda s: shape_len[s]))
    return max(best, key=lambda s: shape_len[s])


rep = ref.groupby('route_id').apply(representative, include_groups=False)
paths = {}
for rid, sid in rep.items():
    s = shapes[shapes['shape_id'] == sid]
    x, y = to_address(s['shape_pt_lat'].to_numpy(), s['shape_pt_lon'].to_numpy())
    paths[rid] = np.round(simplify(np.column_stack([x, y]), SIMPLIFY)).astype(int).tolist()
missing_shape = sorted(set(r.index) - set(paths))
assert not missing_shape, f'Routes without a shape on the reference day: {missing_shape}'
log.info('Shapes: %d routes, %d points', len(paths), sum(len(p) for p in paths.values()))

In [ ]:
# Rings: one per missing bus a weekday. Route averages are rounded by largest remainder, so the rings
# add up to the rounded total.
total = int(round(r['miss'].sum()))
base = np.floor(r['miss']).astype(int)
extra = total - base.sum()
order = (r['miss'] - base).sort_values(ascending=False).index[:extra]
r['rings'] = base
r.loc[order, 'rings'] += 1
assert r['rings'].sum() == total
total

## 5. Analyze

In [ ]:
S = r[['sched', 'seen', 'miss']].sum()
share = S['miss'] / S['sched']
ex = r.groupby('express')[['sched', 'miss']].sum()
daily = kept.groupby('day')[['sched', 'seen']].sum()
daily['share'] = 1 - daily['seen'] / daily['sched']
# without the blank-day rule (never-seen routes still out), for the ledger
raw_share = 1 - rd['seen'].sum() / rd['sched'].sum()
raw_miss = (rd['sched'] - rd['seen']).sum() / len(days)

# South vs north of Roosevelt: every scheduled stop on the reference day (a bus leaving a stop), each
# taking its route's missing share.
st = pd.read_csv(z.open('stops.txt'), dtype=str, usecols=['stop_id', 'stop_lat', 'stop_lon'])
st['x'], st['y'] = to_address(st['stop_lat'].astype(float).to_numpy(), st['stop_lon'].astype(float).to_numpy())
ref_ids = set(ref['trip_id'])
parts = []
for ch in pd.read_csv(z.open('stop_times.txt'), dtype=str, chunksize=2_000_000, usecols=['trip_id', 'stop_id']):
    ch = ch[ch['trip_id'].isin(ref_ids)].merge(ref[['trip_id', 'route_id']], on='trip_id')
    parts.append(ch.groupby(['route_id', 'stop_id']).size())
dep = pd.concat(parts).groupby(level=[0, 1]).sum().rename('n').reset_index().merge(st, on='stop_id')
dep['ghost'] = dep['n'] * dep['route_id'].map(r['share'])
south = dep['y'] < ROOSEVELT
south_stops = dep.loc[south, 'n'].sum() / dep['n'].sum()
south_ghosts = dep.loc[south, 'ghost'].sum() / dep['ghost'].sum()
rate_s = dep.loc[south, 'ghost'].sum() / dep.loc[south, 'n'].sum()
rate_n = dep.loc[~south, 'ghost'].sum() / dep.loc[~south, 'n'].sum()

summary = {
    'days': len(days), 'routes': len(r), 'sched': S['sched'], 'seen': S['seen'], 'miss': S['miss'],
    'rings': total, 'share': share,
    'expressShare': ex.loc[True, 'miss'] / ex.loc[True, 'sched'], 'localShare': ex.loc[False, 'miss'] / ex.loc[False, 'sched'],
    'expressRoutes': int(r['express'].sum()),
    'routeDays': len(rd), 'blankDays': int(rd['blank'].sum()), 'blankRoutes': int(rd.loc[rd['blank'], 'route'].nunique()),
    'rawShare': raw_share, 'rawMiss': raw_miss,
    'worstDay': daily['share'].idxmax(), 'worstShare': daily['share'].max(),
    'bestDay': daily['share'].idxmin(), 'bestShare': daily['share'].min(),
    'southStops': south_stops, 'southGhosts': south_ghosts, 'southRate': rate_s, 'northRate': rate_n,
    'over10': int((r['share'] >= 0.10).sum()), 'under2': int((r['share'] <= 0.02).sum()),
}
for k, v in summary.items():
    print(f'{k:>14}: {v:.4f}' if isinstance(v, float) else f'{k:>14}: {v}')

In [ ]:
def about(n: float) -> str:
    """Round for text: to the nearest ten."""
    return f'about {int(round(n, -1)):,}'


top = r.sort_values('miss', ascending=False).head(3)
finding = (
    f'In October 2026, {about(S["miss"])} of the {int(round(S["sched"], -1)):,} bus trips CTA schedules on a weekday '
    f'never showed up on Bus Tracker ({share:.1%}). South of Roosevelt Road, a bus due at a stop was missing '
    f'{rate_s:.1%} of the time; north of it, {rate_n:.1%}.'
)
print(finding)
print(top[['name', 'miss', 'share']])

## 6. QA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
daily['share'].plot.bar(ax=ax[0], color='0.3', title='Share of scheduled trips never seen, by weekday')
r['share'].plot.hist(bins=30, ax=ax[1], color='0.3', title='Routes by share never seen')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
for rid, p in paths.items():
    p = np.array(p)
    ax.plot(p[:, 0], p[:, 1], lw=0.4, color='0.7')
ax.scatter(dep['x'], dep['y'], s=dep['ghost'] * 0.4, color='k')
ax.axhline(ROOSEVELT, lw=0.5)
ax.set_aspect('equal'); ax.set_title('Missing stop departures (dots) on route shapes')

## 7. Export

In [ ]:
ledger = Ledger(
    source='CTA Bus Tracker vehicle positions, Ghost Buses archive (Chi Hack Night; Mansueto Institute)',
    dataset_id='full_day_data', url=ARCHIVE_INFO, pulled=PULLED,
    units='Trips per average weekday, by route',
    method=['Scheduled: GTFS trips per route and day', 'Seen: distinct Bus Tracker trips per route and day',
            f'Route-days under {BLANK:.0%} seen left out as blank', 'Routes never seen left out'],
    caveats=['A bus whose tracker is off counts as missing', 'Trips counted, not matched by id'],
)
feed_date = max(date(*i.date_time[:3]) for i in z.infolist() if i.filename == 'stop_times.txt')
route_rows = []
for rid, v in r.sort_values('miss', ascending=False).iterrows():
    route_rows.append({
        'id': rid, 'name': v['name'], 'express': bool(v['express']),
        'sched': round(v['sched'], 1), 'seen': round(v['seen'], 1), 'miss': round(v['miss'], 1),
        'share': round(v['share'], 4), 'rings': int(v['rings']), 'days': int(v['days']), 'blank': int(v['blank']),
        'path': paths[rid],
    })
payload = {
    'window': [days[0], days[-1]], 'daysList': days, 'missingDays': missing_days,
    'blankRule': BLANK,
    'never': [{'id': k, 'name': bus.loc[k, 'route_long_name']} for k in never],
    'gtfs': {'file': GTFS_ZIP.name, 'snapshot': GTFS_ZIP.stem.split('_')[-1], 'feedDate': feed_date.isoformat(), 'refDay': REF_DAY.isoformat(), 'url': GTFS_URL},
    'sources': {'archive': ARCHIVE_INFO, 'busTracker': BUSTRACKER_URL},
    'summary': {k: (round(v, 4) if isinstance(v, float) else v) for k, v in summary.items()},
    'daily': [{'day': d, 'sched': int(v['sched']), 'seen': int(v['seen']), 'share': round(v['share'], 4)} for d, v in daily.iterrows()],
    'routes': route_rows,
}
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=finding, payload=payload)
log.info('Wrote %s (%.0f KB)', OUT_PATH, OUT_PATH.stat().st_size / 1024)